In [ ]:
def extended_euclidean(a, b):
    if b == 0:
        return a, 1, 0   # gcd, x, y
    else:
        gcd, x1, y1 = extended_euclidean(b, a % b)

        x = y1
        y = x1 - (a // b) * y1

        return gcd, x, y



In [ ]:
def encrypt(text, key):
    result = ""
    for char in text:
        if char.isalpha():
            shift = key % 26
            if char.isupper():
                result += chr((ord(char) - 65 + shift) % 26 + 65)
            else:
                result += chr((ord(char) - 97 + shift) % 26 + 97)
        else:
            result += char   # preserve spaces/special characters
    return result


def decrypt(text, key):
    result = ""
    for char in text:
        if char.isalpha():
            shift = key % 26
            if char.isupper():
                result += chr((ord(char) - 65 - shift) % 26 + 65)
            else:
                result += chr((ord(char) - 97 - shift) % 26 + 97)
        else:
            result += char
    return result


# ---- Input ----
plaintext = input("Enter plaintext: ")
key = int(input("Enter key (0-25): "))

ciphertext = encrypt(plaintext, key)
decrypted = decrypt(ciphertext, key)

print("\n--- Task 1 Output ---")
print("Plaintext :", plaintext)
print("Ciphertext:", ciphertext)
print("Decrypted :", decrypted)

from collections import Counter

def frequency_analysis(ciphertext):
    filtered = [c.upper() for c in ciphertext if c.isalpha()]
    freq = Counter(filtered)

    print("\n--- Frequency Count ---")
    for char, count in freq.items():
        print(f"{char}: {count}")

    most_common = freq.most_common(1)[0][0]
    print("\nMost frequent character:", most_common)

    # Assume mapping with E, T, A
    possible_keys = []
    for common in ['E', 'T', 'A']:
        key = (ord(most_common) - ord(common)) % 26
        possible_keys.append(key)

    print("\n--- Possible Keys (E, T, A assumption) ---")
    for key in possible_keys:
        print(f"Key {key}: {decrypt(ciphertext, key)}")


# Input for frequency analysis
cipher_input2 = input("\nEnter ciphertext for frequency analysis: ")
frequency_analysis(cipher_input2)

In [ ]:
def create_matrix(text, key_length, pad_char='X'):

    rows = len(text) // key_length
    if len(text) % key_length != 0:
        rows += 1
        text += pad_char * (rows * key_length - len(text))

    matrix = []
    for r in range(rows):
        row = list(text[r*key_length:(r+1)*key_length])
        matrix.append(row)
    return matrix


def encrypt(plaintext, key):
    plaintext = plaintext.replace(" ", "").upper()
    key_length = len(key)
    matrix = create_matrix(plaintext, key_length)

    key_order = sorted(list(enumerate(key)), key=lambda x: x[1])

    ciphertext = ""
    for index, _ in key_order:
        for row in matrix:
            ciphertext += row[index]

    return ciphertext


def decrypt(ciphertext, key):
    key_length = len(key)
    text_length = len(ciphertext)
    rows = text_length // key_length

    key_order = sorted(list(enumerate(key)), key=lambda x: x[1])
    matrix = [['']*key_length for _ in range(rows)]

    k = 0
    for index, _ in key_order:
        for r in range(rows):
            matrix[r][index] = ciphertext[k]
            k += 1

    plaintext = ""
    for row in matrix:
        plaintext += "".join(row)

    return plaintext

In [ ]:
import os
import time
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization, hashes


def generate_symmetric_key():
    key = Fernet.generate_key()
    with open("symmetric.key", "wb") as f:
        f.write(key)
    return key


def encrypt_file_symmetric(filename, key):
    fernet = Fernet(key)

    with open(filename, "rb") as file:
        data = file.read()

    encrypted = fernet.encrypt(data)

    with open("encrypted_sym.txt", "wb") as file:
        file.write(encrypted)


def decrypt_file_symmetric(filename, key):
    fernet = Fernet(key)

    with open(filename, "rb") as file:
        encrypted_data = file.read()

    decrypted = fernet.decrypt(encrypted_data)

    with open("decrypted_sym.txt", "wb") as file:
        file.write(decrypted)


def generate_rsa_keys():
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )

    public_key = private_key.public_key()

    return private_key, public_key


def encrypt_file_rsa(filename, public_key):
    with open(filename, "rb") as f:
        data = f.read()

    encrypted = public_key.encrypt(
        data,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

    with open("encrypted_rsa.bin", "wb") as f:
        f.write(encrypted)


def decrypt_file_rsa(filename, private_key):
    with open(filename, "rb") as f:
        encrypted_data = f.read()

    decrypted = private_key.decrypt(
        encrypted_data,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

    with open("decrypted_rsa.txt", "wb") as f:
        f.write(decrypted)

In [ ]:
import hashlib
import hmac

key = b'secret_key'
file_name = "file.txt"

# Function to compute SHA-512 and HMAC
def compute_hash_hmac(filename):
    with open(filename, 'rb') as f:
        data = f.read()

    sha512_hash = hashlib.sha512(data).hexdigest()
    hmac_val = hmac.new(key, data, hashlib.sha512).hexdigest()

    return sha512_hash, hmac_val


with open(file_name, 'w') as f:
    f.write("This is original content.\n")

print("BEFORE MODIFICATION")
hash1, hmac1 = compute_hash_hmac(file_name)
print("SHA-512:", hash1)
print("HMAC:", hmac1)

with open(file_name, 'a') as f:
    f.write("This line was added (tampering).\n")

print("\nAFTER MODIFICATION")
hash2, hmac2 = compute_hash_hmac(file_name)
print("SHA-512:", hash2)
print("HMAC:", hmac2)

print("\nCOMPARISON")
if hash1 != hash2:
    print("Hash changed -> File Modified!")
else:
    print("Hash unchanged")

if hmac1 != hmac2:
    print("HMAC changed -> Integrity + Authentication Failed!")
else:
    print("HMAC unchanged")